# Strands Agents 3 - Multi-Agent & a Research-Assistant Capstone

One agent can only do so much. Here we build teams: an orchestrator with specialist sub-agents, an autonomous swarm, a deterministic graph, and a workflow tool - then combine everything into a research assistant inspired by the AWS drug-discovery example.

**Series:** these notebooks go scratch -> foundations -> intermediate. Run them in order, top to bottom. Every cell here is real, runnable code (verified against `strands-agents` 1.42).

## 0 · What you need (one-time)

1. **Python 3.10+**.
2. An **AWS account** with **Amazon Bedrock model access** enabled for the Claude models we use. In the AWS console: *Bedrock -> Model access -> Enable* **Claude Haiku 4.5** (and optionally **Claude Sonnet 4**). Access is per-region.
3. **AWS credentials** with permission to call Bedrock (`bedrock:InvokeModel` and `bedrock:InvokeModelWithResponseStream`).

You will set the region and credentials two cells down. Nothing else to install by hand - the next cell does it.

### Install the SDK

`%pip` installs into the *kernel that is running this notebook*, so it works the same in Google Colab and in a local VS Code / Jupyter venv.

In [ ]:
%pip install -q strands-agents strands-agents-tools
print("installed")

### Configure region + credentials

This cell works in **both** environments:

- **Google Colab:** open the **Secrets** panel (the key icon in the left sidebar), add `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` (and `AWS_SESSION_TOKEN` if you use temporary keys), then run the cell.
- **VS Code / local:** you usually do not need to paste keys here at all - if you have already run `aws configure` or exported `AWS_` variables in your shell, Strands picks them up automatically. The cell just sets your region.

> Set `AWS_DEFAULT_REGION` to the region where you enabled model access.

In [1]:
import os
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"   # <-- match your enabled-model region

try:
    from google.colab import userdata           # only exists on Colab
    os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
    # os.environ["AWS_SESSION_TOKEN"]   = userdata.get("AWS_SESSION_TOKEN")  # if temporary creds
    print("Colab: credentials loaded from Secrets.")
except ModuleNotFoundError:
    print("Local: using your existing AWS credentials (aws configure / env vars).")
print("Region:", os.environ["AWS_DEFAULT_REGION"])

Local: using your existing AWS credentials (aws configure / env vars).
Region: us-east-1


### Choose a model

Strands talks to Bedrock through a `BedrockModel`. Two things to notice:

- **The `us.` prefix.** Newer Claude models are served on-demand only through a *cross-region inference profile*. The id therefore starts with `us.` (or `global.` / `eu.`). Plain `anthropic.claude-...` would raise *"on-demand throughput isn't supported"*.
- **We set timeouts + adaptive retries** on the underlying boto client. The `adaptive` retry mode automatically backs off and retries when Bedrock throttles you - the single most common error under classroom / burst load.

We default to **Haiku 4.5** because it is fast and cheap, which is ideal while learning. Swap in **Sonnet 4** for harder reasoning by changing one argument.

In [2]:
from strands import Agent, tool
from strands.models import BedrockModel
from botocore.config import Config

MODEL_HAIKU  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # fast + cheap (default)
MODEL_SONNET = "us.anthropic.claude-sonnet-4-20250514-v1:0"    # stronger reasoning

def get_model(model_id=MODEL_HAIKU, temperature=0.3, max_tokens=2048):
    """Build a configured Bedrock model. Reuse this everywhere."""
    return BedrockModel(
        model_id=model_id,
        temperature=temperature,        # 0 = deterministic, higher = more varied
        max_tokens=max_tokens,          # cap on the model's reply length
        boto_client_config=Config(
            read_timeout=900, connect_timeout=900,
            retries={"max_attempts": 4, "mode": "adaptive"},   # auto-handle throttling
        ),
    )

MODEL = get_model()

# Small helper: pull clean text out of an AgentResult.
# result.message looks like {"role": "assistant", "content": [{"text": "..."}]}
def say(result):
    return "".join(block.get("text", "") for block in result.message["content"]).strip()

print("Model ready:", MODEL.get_config()["model_id"])

Model ready: us.anthropic.claude-haiku-4-5-20251001-v1:0


### Smoke test (your first real Bedrock call)

If your access, credentials, and region are correct, this prints **setup OK**. If it errors, read the message:

- *AccessDenied / could not load credentials* -> credentials/region not set (cell above).
- *on-demand throughput isn't supported* or *model id ... not found* -> your account exposes this model under a different inference-profile prefix; try `global.` instead of `us.`, or run `aws bedrock list-inference-profiles` to see what is available.

In [3]:
smoke = Agent(model=MODEL, callback_handler=None)   # callback_handler=None => no streaming print
print(say(smoke("Reply with exactly: setup OK")))

setup OK


---
## 1 · Why multiple agents, and the five patterns

Splitting work across focused agents beats one giant agent: each gets its own role, tools, and even its own model. The question is always *how control and data flow*. Strands gives five patterns:

| Pattern | Who decides the path | How data is handed off | Use when |
|---|---|---|---|
| **Agents as Tools** | an orchestrator you write | request string in, text out | clear manager + specialists |
| **Swarm** | the agents (emergent) | `handoff_to_agent` + shared context | path not known up front |
| **Graph** | edges you define | each node's output feeds the next | deterministic, repeatable |
| **Workflow** | task dependencies | task outputs feed dependent tasks | a fixed pipeline as one tool |
| **A2A** | across processes / orgs | Agent Card + JSON-RPC over HTTP | agents on different machines |

The first four run in-process and are demonstrated live below. A2A is networked, so we show it as a two-terminal recipe at the end.

> These cells make several model calls each. We stay on Haiku 4.5 to keep them fast and cheap.

## 2 · Pattern 1 - Agents as Tools (an orchestrator + specialists)

Wrap a specialist agent inside a `@tool`. The orchestrator then "calls" the specialist like any other tool: it passes a **request string** and gets back the specialist's **text answer**. That string-in/string-out is the handoff.

In [4]:
@tool
def math_expert(question: str) -> str:
    """Solve a math problem. Use for anything numeric or symbolic."""
    specialist = Agent(model=MODEL, system_prompt="You are a precise mathematician. Show the result clearly.",
                       tools=[calculator], callback_handler=None)
    return say(specialist(question))

@tool
def writing_expert(request: str) -> str:
    """Write or polish prose. Use for explanations, summaries, copy."""
    specialist = Agent(model=MODEL, system_prompt="You are a crisp technical writer.", callback_handler=None)
    return say(specialist(request))

from strands_tools import calculator
orchestrator = Agent(
    model=MODEL,
    system_prompt="You route each part of a request to the best specialist tool, then combine the results.",
    tools=[math_expert, writing_expert],
    callback_handler=None,
)
print(say(orchestrator("Compute 144*12, then write a one-line caption celebrating the number.")))

╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 144 * 12            │                                                                            │
│  │ Result    │ 1728                │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Perfect! Here are the results:

**Computation:** 144 × 12 = **1,728**

**Caption:** 1728: The Perfect Cube—12³ Equals Mathematical Elegance.


## 3 · Pattern 2 - Swarm (autonomous handoff)

A **Swarm** is a team with no central router. Each agent gets a built-in `handoff_to_agent` tool and decides when to pass control. A shared context travels with the handoff, so the next agent sees the message, the original task, and prior work. The path is *emergent*.

Each agent needs a `name` and a `description` so teammates know who to hand to.

In [5]:
from strands.multiagent import Swarm

researcher = Agent(name="researcher", model=MODEL,
    description="Gathers and lists key facts about a topic.",
    system_prompt="List 3 concise factual bullet points, then hand off to the writer.",
    callback_handler=None)

writer = Agent(name="writer", model=MODEL,
    description="Turns facts into a short, polished paragraph.",
    system_prompt="Write one tight paragraph from the facts you receive. Then finish.",
    callback_handler=None)

swarm = Swarm([researcher, writer], entry_point=researcher, max_handoffs=6)
result = swarm("Explain why bumblebees can fly, for a general audience.")
print(result.results[writer.name].result.message["content"][0]["text"]
      if hasattr(result, "results") else result)

Here's a polished paragraph for a general audience:

Bumblebees defy the odds of physics through a combination of remarkable adaptations that allow their stocky bodies to take flight. Their wings beat an astonishing 130 to 240 times per second, moving in rapid figure-8 patterns rather than simple up-and-down flaps—a motion that generates lift far more efficiently than you might expect. What makes this feat even more impressive is that their wings are actually quite small relative to their chunky bodies, yet their flexible wing structure and articulated joints compensate beautifully for this apparent disadvantage. The secret lies in aerodynamic wizardry: as their wings slice through the air, they create tiny whirlwinds or vortices that produce extra lift, allowing these fuzzy pollinators to hover, maneuver, and carry heavy loads of pollen despite their relatively heavy bodies. In essence, bumblebees don't just flap their way through the air—they harness the physics of motion to achieve 

> `swarm(...)` returns a multi-agent result. Above we read the final writer node's text. If the structure differs in your version, `print(result)` shows everything the team produced.

## 4 · Pattern 3 - Graph (you wire the path)

A **Graph** runs agents as **nodes** along **edges** you define. There is no handoff tool: Strands automatically feeds each node's output into the nodes it points to. Deterministic and auditable.

We set a max-execution cap as a safety net (graphs *can* be built with loops).

In [6]:
from strands.multiagent import GraphBuilder

outline = Agent(name="outline", model=MODEL,
    system_prompt="Produce a 3-bullet outline for the topic.", callback_handler=None)
draft = Agent(name="draft", model=MODEL,
    system_prompt="Write a short paragraph from the outline you receive.", callback_handler=None)

gb = GraphBuilder()
gb.add_node(outline, "outline")
gb.add_node(draft, "draft")
gb.add_edge("outline", "draft")     # outline's output flows into draft
gb.set_entry_point("outline")
gb.set_max_node_executions(10)      # safety cap
graph = gb.build()

g_result = graph("The benefits of cycling to work.")
print(g_result.results["draft"].result.message["content"][0]["text"]
      if hasattr(g_result, "results") else g_result)

# The Benefits of Cycling to Work

Cycling to work offers a powerful combination of personal and environmental advantages that make it an ideal commute choice. Beyond the obvious health benefits—such as improved cardiovascular fitness, increased muscle strength, and calorie burning—cycling provides significant financial savings by eliminating fuel costs and reducing vehicle maintenance expenses. Perhaps most importantly, the mental health benefits are substantial; the combination of outdoor exercise, fresh air, and physical activity triggers endorphin release, naturally reducing stress and anxiety while boosting mood and mental clarity. This energizing start to your day translates directly into improved focus and productivity at work, making cycling not just a healthier choice for your body, but a smarter investment in your overall well-being and professional performance.


## 5 · Pattern 4 - Workflow (a fixed pipeline as one tool)

`workflow` is a ready-made tool from `strands-agents-tools`. You describe tasks and their dependencies in plain language; it captures each task's output and passes it to dependent tasks, running independent ones in parallel. Use it to package a standard, repeatable process behind a single agent action.

In [7]:
from strands_tools import workflow

flow_agent = Agent(model=MODEL, tools=[workflow], callback_handler=None)
print(say(flow_agent(
    "Create and run a workflow with two steps: "
    "1) list three healthy lunch ideas; "
    "2) turn step 1's list into a short shopping list. Step 2 depends on step 1."
)))

I don't need to use any tools for this request. Here are three healthy lunch ideas:

1. **Grilled Chicken & Quinoa Bowl**
   A nutritious bowl featuring grilled chicken breast, fluffy quinoa, roasted vegetables (like broccoli and bell peppers), and a light lemon-tahini dressing. This meal is high in protein, fiber, and essential nutrients, keeping you satisfied and energized throughout the afternoon.

2. **Mediterranean Chickpea Salad**
   A colorful salad combining chickpeas, cherry tomatoes, cucumber, red onion, feta cheese, and fresh herbs (parsley and mint) tossed with olive oil and lemon juice. It's packed with plant-based protein, healthy fats, and vitamins, making it both filling and refreshing.

3. **Turkey & Avocado Wrap**
   A whole wheat wrap filled with sliced turkey breast, creamy avocado, fresh spinach, tomato, and a spread of hummus. This balanced meal provides lean protein, healthy fats, and fiber, making it perfect for a quick, portable lunch that doesn't sacrifice nut

Tool 'file_write' not found in parent agent's tool registry


# Consolidated Healthy Lunch Shopping List

## PROTEINS
- [ ] Boneless, skinless chicken breasts – 1.5 lbs
- [ ] Ground turkey (or turkey breast slices) – 1 lb
- [ ] Canned chickpeas – 2 cans (15 oz each)
- [ ] Eggs – 1 dozen (for optional toppings)

## VEGETABLES
- [ ] Cherry tomatoes – 2 pints
- [ ] Cucumbers – 2 medium
- [ ] Red bell peppers – 2
- [ ] Red onion – 1 medium
- [ ] Spinach or mixed greens – 1 large container (5-6 oz)
- [ ] Avocados – 3-4
- [ ] Carrots – 1 lb (fresh or pre-cut)
- [ ] Celery – 1 bunch (optional, for crunch)

## GRAINS & WRAPS
- [ ] Quinoa – 1 lb (uncooked)
- [ ] Whole wheat tortillas – 1 package (8-10 count)
- [ ] Brown rice or farro – 1 lb (optional alternative)

## DAIRY & ALTERNATIVES
- [ ] Feta cheese – 8 oz block
- [ ] Greek yogurt – 1 container (32 oz)
- [ ] Cheddar or Swiss cheese slices – 1 package (optional)

## PANTRY ITEMS
- [ ] Extra virgin olive oil
- [ ] Balsamic vinegar
- [ ] Red wine vinegar
- [ ] Dijon mustard
- [ ] Honey – 1 jar (if need

Perfect! Here's the **Consolidated Shopping List** for all three healthy lunches:

## 🛒 SHOPPING LIST - All Three Lunches

### 🍗 **PROTEINS**
- Chicken breast (1.5 lbs) - for Grilled Chicken & Quinoa Bowl
- Turkey breast, sliced (1 lb) - for Turkey & Avocado Wrap
- Chickpeas, canned (2 cans, 15 oz each) - for Mediterranean Chickpea Salad

### 🥬 **FRESH VEGETABLES**
- Broccoli (1 head) - for Grilled Chicken & Quinoa Bowl
- Bell peppers (2-3, mixed colors) - for Grilled Chicken & Quinoa Bowl
- Cherry tomatoes (1 pint) - for Mediterranean Chickpea Salad
- Cucumber (1 large) - for Mediterranean Chickpea Salad
- Red onion (1 medium) - for Mediterranean Chickpea Salad
- Fresh spinach (1 bunch or 5 oz bag) - for Turkey & Avocado Wrap
- Tomato (2 medium) - for Turkey & Avocado Wrap

### 🥑 **FRESH PRODUCE**
- Avocado (2) - for Turkey & Avocado Wrap

### 🌾 **GRAINS & WRAPS**
- Quinoa (1 lb dry) - for Grilled Chicken & Quinoa Bowl
- Whole wheat wraps (1 package) - for Turkey & Avocado Wrap

### 🧀

Excellent! Your workflow has completed successfully! Here's what happened:

## Workflow Summary

✅ **Both tasks completed successfully!**

### Task 1: Lunch Ideas
The first task generated three healthy lunch ideas.

### Task 2: Shopping List
The second task took the lunch ideas from Step 1 and created a consolidated shopping list of ingredients needed for all three lunches.

The workflow demonstrates proper dependency management - Task 2 waited for Task 1 to complete before executing, ensuring it had the lunch ideas to work with before creating the shopping list.

Would you like me to show you the detailed output from each task, or would you like to create another workflow?


## 6 · Pattern 5 - A2A (across machines) — recipe, not a single cell

A2A is for agents owned by *different* teams or running as *separate* services, talking over HTTP. It needs an extra install and two processes, so it does not fit one notebook cell. Here is the exact, correct recipe.

**Install the extra:**
```bash
pip install 'strands-agents[a2a]' strands-agents-tools
```

**Terminal 1 - serve an agent (`server.py`):**
```python
from strands import Agent
from strands.multiagent.a2a import A2AServer

agent = Agent(name="Calculator", description="Does arithmetic.")
A2AServer(agent=agent, port=9000).serve()   # Agent Card at /.well-known/agent.json
```

**Terminal 2 - a client agent that discovers and uses it (`client.py`):**
```python
from strands import Agent
from strands_tools.a2a_client import A2AClientToolProvider

provider = A2AClientToolProvider(known_agent_urls=["http://localhost:9000"])
agent = Agent(tools=provider.tools)
print(agent("Use the calculator agent to compute 12 * 34."))
```

The server publishes an **Agent Card** (its skills); the client discovers it and sends **JSON-RPC over HTTP**. That is the handoff across a network boundary.

---
## 7 · Capstone — a research assistant

We combine what we have learned into the architecture from the AWS *drug-discovery research assistant* blog, simplified to run with **only Bedrock** (no external API keys):

- **Specialist sub-agents as tools:** a `planning_agent` (decides what to look up) and a `synthesis_agent` (writes the final report).
- **"Database" tools:** `search_papers` and `search_web`. We return small built-in datasets so the notebook runs offline. In production these would be **MCP servers** (arXiv, PubMed, web search) wired in exactly as in Notebook 2.
- **An orchestrator** that holds it together with a **sliding-window memory**, deciding which tools to use and in what order. You do not script the steps - it plans them.

### Step 7a - the data-source tools (mocked)

In [8]:
@tool
def search_papers(topic: str) -> str:
    """Search academic papers for a topic. Returns titles + one-line summaries."""
    DB = {
        "sleep": [
            ("Short sleep and metabolism (2023)", "Links <6h sleep to insulin resistance."),
            ("Sleep and memory consolidation (2021)", "Deep sleep strengthens new memories."),
        ],
        "exercise": [
            ("Zone-2 training review (2022)", "Low-intensity volume improves mitochondrial density."),
            ("Resistance training and aging (2020)", "Lifting preserves muscle mass after 50."),
        ],
    }
    hits = DB.get(topic.lower())
    if not hits:
        return f"No papers indexed for '{topic}'. Known topics: {list(DB)}."
    return "\n".join(f"- {t}: {s}" for t, s in hits)

@tool
def search_web(query: str) -> str:
    """Search the web for recent, general-audience information on a query."""
    NEWS = {
        "sleep": "Recent coverage stresses consistent sleep/wake times over total hours.",
        "exercise": "Recent coverage highlights 'exercise snacks' - short movement breaks - for desk workers.",
    }
    for k, v in NEWS.items():
        if k in query.lower():
            return v
    return "No recent general coverage found for that query."

### Step 7b - the specialist sub-agents (as tools)

In [9]:
@tool
def planning_agent(question: str) -> str:
    """Analyze a research question and list which sources to consult and what to search for."""
    planner = Agent(model=MODEL, system_prompt=(
        "You are a research planner. Given a question, output a short numbered plan: "
        "which tools to call (search_papers, search_web) and the exact search terms. Be specific and brief."
    ), callback_handler=None)
    return say(planner(question))

@tool
def synthesis_agent(findings: str) -> str:
    """Turn raw findings into a structured report: Summary, Key Points, Sources."""
    writer = Agent(model=MODEL, system_prompt=(
        "You are a research synthesizer. Write a concise report with three sections: "
        "## Summary (2-3 sentences), ## Key Points (bullets), ## Sources (what was used). "
        "Only use the findings provided; do not invent facts."
    ), callback_handler=None)
    return say(writer(findings))

### Step 7c - the orchestrator

It gets every tool plus a `SlidingWindowConversationManager`, and a prompt describing the phases. Then we just ask it a question.

In [10]:
from strands.agent.conversation_manager import SlidingWindowConversationManager

research_assistant = Agent(
    model=MODEL,
    system_prompt=(
        "You are a research assistant. For a research request:\n"
        "1) call planning_agent to make a plan;\n"
        "2) call search_papers and search_web as the plan suggests;\n"
        "3) call synthesis_agent with everything you gathered to produce the final report.\n"
        "Answer simple questions directly without tools."
    ),
    tools=[planning_agent, search_papers, search_web, synthesis_agent],
    conversation_manager=SlidingWindowConversationManager(window_size=20),
    callback_handler=None,
)

report = research_assistant("Write a short report on how sleep affects health.")
print(say(report))

## Report: How Sleep Affects Health

### Summary
Sleep is a fundamental biological process that significantly impacts overall health and well-being. Research demonstrates that both the duration and consistency of sleep play critical roles in maintaining physical and mental health. Insufficient sleep—particularly less than 6 hours per night—is linked to metabolic dysfunction, while maintaining regular sleep schedules appears equally important as total sleep hours.

### Key Health Effects

**Metabolic Health**
Sleep deprivation directly affects metabolic function. Studies show that sleeping less than 6 hours is associated with insulin resistance, which can increase the risk of type 2 diabetes and weight gain. Adequate sleep helps regulate hormones that control appetite and glucose metabolism.

**Cognitive Function and Memory**
Deep sleep is essential for memory consolidation and learning. During sleep, the brain processes and strengthens new memories, making quality sleep crucial for aca

The assistant planned, pulled from both sources, and synthesized a report - and you never hard-coded the order. Check the work it did:

In [11]:
s = report.metrics.get_summary()
print("Model calls (cycles):", s["total_cycles"])
print("Tools used         :", list(s.get("tool_usage", {}).keys()))
print("Total tokens       :", s["accumulated_usage"]["totalTokens"])

Model calls (cycles): 5
Tools used         : ['planning_agent', 'search_papers', 'search_web', 'synthesis_agent']
Total tokens       : 8466


### Step 7d - a typed summary on top (combining patterns)

Finally, get a machine-usable summary of that report with structured output - so a downstream system could store or route it.

In [12]:
from pydantic import BaseModel, Field

class ResearchSummary(BaseModel):
    topic: str = Field(description="the research topic")
    key_findings: list[str] = Field(description="3-5 short findings")
    confidence: str = Field(description="one of: low, medium, high")

summary_agent = Agent(model=MODEL, callback_handler=None)
typed = summary_agent.structured_output(
    ResearchSummary,
    "Summarize this research report into structured fields:\n\n" + say(report),
)
print("topic        :", typed.topic)
print("confidence   :", typed.confidence)
for f in typed.key_findings:
    print("  -", f)

/var/folders/0s/szt9vrsx5px2yt75kmpf4_j40000gn/T/ipykernel_19959/813360388.py:9: DeprecationWarning: Agent.structured_output method is deprecated. You should pass in `structured_output_model` directly into the agent invocation. see: https://strandsagents.com/latest/documentation/docs/user-guide/concepts/agents/structured-output/
  typed = summary_agent.structured_output(


topic        : How Sleep Affects Health
confidence   : high
  - Sleep deprivation of less than 6 hours per night is linked to insulin resistance, type 2 diabetes risk, and weight gain
  - Deep sleep is essential for memory consolidation, learning, and cognitive function including academic and work performance
  - Maintaining consistent sleep and wake times is as important as total sleep duration for regulating circadian rhythms and physiological health
  - Chronic sleep deprivation weakens immune function and increases risk of cardiovascular problems and chronic diseases
  - Recommended sleep duration is 6-8 hours per night with consistent schedules and good sleep hygiene practices


---
## Where to go from here

- **Make it real:** replace `search_papers` / `search_web` with MCP servers (Notebook 2's pattern) - arXiv, PubMed, a web-search API.
- **Add memory across sessions:** give the orchestrator a `FileSessionManager` (dev) or `S3SessionManager` (prod) so it remembers a user between runs.
- **Ship it:** wrap the orchestrator in an AgentCore app and deploy:

```python
from bedrock_agentcore.runtime import BedrockAgentCoreApp
app = BedrockAgentCoreApp()

@app.entrypoint
def invoke(payload, context):
    return {"result": str(research_assistant(payload.get("prompt")))}

app.run()
# then:  agentcore configure -e app.py   ->   agentcore launch   ->   agentcore invoke '{"prompt": "..."}'
```

You have gone from a three-line agent to a deployable multi-agent system. That is the whole arc.